**Install:** `pip install -U langchain langchain-openai langgraph`

# 🎯 8. ReAct and Tool-Using Agents

**ReAct** (Reasoning + Acting) is the most widely-used agent architecture. It combines step-by-step reasoning with tool execution in a structured loop.

In this notebook:

1. **The ReAct paper** — key insights
2. **Thought → Action → Observation** cycle
3. **ReAct from scratch** — pure Python implementation
4. **ReAct with LangGraph** — production implementation
5. **Trace inspection** — debugging agent reasoning
6. **Failure modes** — infinite loops, wrong tools

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

client = OpenAI()
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
print(f'Model: {LLM_MODEL}')

## 8.1 The ReAct Pattern

The ReAct paper (Yao et al., 2022) showed that combining **reasoning** traces with **actions** significantly improves agent performance.

```
Thought: I need to find the population of Tokyo.
Action: search("Tokyo population 2024")
Observation: Tokyo has approximately 13.96 million people.
Thought: Now I have the answer.
Action: finish("Tokyo has approximately 13.96 million people.")
```

### Why ReAct Works

| Component | Without ReAct | With ReAct |
|-----------|-------------|------------|
| Reasoning | Hidden/implicit | Explicit thoughts |
| Actions | Fixed pipeline | Dynamic tool selection |
| Observations | Not used | Fed back to improve reasoning |
| Debugging | Black box | Full trace available |

## 8.2 Implementing ReAct from Scratch

Let's build a complete ReAct agent without any framework:

<img src="images/react-without-framework.png" width="40%" style="border-radius:10px;margin:12px 0;"/>


## 8.3 ReAct with LangGraph — Production Implementation

LangGraph provides the same pattern with production features: checkpointing, error handling, and automatic tool execution.

<img src="images/react-with-langgraph.png" width="50%" style="border-radius:10px;margin:12px 0;"/>

## 8.4 Failure Modes and Mitigations

ReAct agents can fail in several ways:

| Failure | Cause | Mitigation |
|---------|-------|------------|
| **Infinite loop** | Agent keeps calling tools without converging | Max step limit, loop detection |
| **Wrong tool** | Agent picks inappropriate tool | Better descriptions, fewer tools |
| **Hallucinated observation** | LLM generates fake tool results | `stop=['Observation:']` in generation |
| **Over-reliance on tools** | Agent calls tools for trivial questions | System prompt: "only use tools when necessary" |
| **Error cascade** | Tool error causes agent to spiral | Error handling in tool execution |

The `stop=['Observation:']` trick (from HuggingFace's agents course) is critical: it prevents the LLM from hallucinating tool results by stopping generation right before the observation, allowing real tool output to be injected.

## 💡 Exercise 8: Build a Research ReAct Agent

**Task**: Build a ReAct agent with these tools:
1. `web_search(query)` — simulated web search
2. `calculator(expression)` — math evaluation
3. `note_taker(note)` — saves notes to a list (memory)
4. `read_notes()` — reads saved notes

The agent should be able to research a topic, take notes, do calculations, and synthesize findings.

In [ ]:
# Exercise 8: YOUR CODE HERE


## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **ReAct** | Thought → Action → Observation in a loop |
| **Stop tokens** | Stop before `Observation:` to prevent hallucinated tool results |
| **LangGraph ReAct** | agent → conditional_edge → ToolNode → agent loop |
| **Checkpointing** | MemorySaver persists the full trace for debugging |
| **Failure modes** | Infinite loops, wrong tools, hallucinated observations — all preventable |

### What's Next

In **Notebook 09: Single-Agent Architectures**, we design complete, modular agent systems — combining tools, memory, planning, and ReAct into production-ready architectures.